# Loading

In [ ]:

%pip install matplotlib pandas pyfixest statsmodels seaborn geopandas adjustText jinja2

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import os
import geopandas as gpd
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import argparse
from pathlib import Path
import seaborn as sns
import statsmodels.api as sm
import pyfixest as pf
from adjustText import adjust_text
from matplotlib.colors import Normalize,BoundaryNorm

K = -1
merge_idf = True



# # Set the width of your LaTeX document in points
document_width_pt = 511.  # Adjust this according to your LaTeX template
#plt.rc('text', usetex=True)
plt.rc('font', family='serif')
toulouse_color = (132/255, 46/255, 27/255)
missing_color = (0/255, 0/255, 0/255)
label_size = 18
font_size = 15  # Adjust according to your preference
plt.rcParams.update({
    "font.size": font_size,
    "axes.labelsize": font_size,
    "axes.titlesize": font_size+2,
    "xtick.labelsize": font_size-3,
    "ytick.labelsize": font_size-3,
    "legend.fontsize": font_size-3,
    "legend.title_fontsize": font_size,
    "figure.titlesize": font_size,
})


from matplotlib.colors import Normalize,BoundaryNorm
def plot_map(df, col_name, ax, fig, vmin=None, vmax=None, ze_shocked=None, add_cbar=True, norm=None, cmap="viridis",fmt_colorbar = '%1.4f',missing_color = "grey",title_cbar = None):
    
    tmp = france.merge(df, on="ze2010", how="left")
    
    tmp[col_name].fillna(0, inplace=True)
    #return tmp
    
    if vmin is None:
        vmin = df.query(f'{col_name} > 0')[col_name].min()
    if vmax is None:
        vmax = df[col_name].max()
    
    if norm is None:
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

    if tmp.query(f' {col_name} == 0').shape[0] > 0:
        tmp.query(f'{col_name} == 0').plot(facecolor=missing_color, ax=ax, edgecolor="black")
    tmp.query(f'{col_name} > 0').plot(column=col_name, ax=ax, norm=norm, cmap=cmap, edgecolor="black")
    
    # Inset for IDF
    if not merge_idf:
        ax_inset = inset_axes(ax, width="65%", height="65%", loc="upper right",
                              bbox_to_anchor=(0.745, 0.775, 0.25, 0.25), 
                              bbox_transform=ax.transAxes, borderpad=1)
        data_idf = tmp[tmp['ze2010'].isin(idf_ze)]
        
        if data_idf.shape[0] > 0 :data_idf.query(f'{col_name} > 0').plot(column=col_name, ax=ax_inset, norm=norm, cmap=cmap, edgecolor="black")
        ax_inset.set_xticks([])
        ax_inset.set_yticks([])
    if ze_shocked is not None:
        for ze in ze_shocked:
            tmp.query(f'ze2010 == "{ze}"').plot(facecolor="none", ax=ax, edgecolor="red", hatch="//")

        
    ax.set_xlim(-5, 10)
    ax.set_ylim(42, 52)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.axis('off')  # <-- add this line
    if add_cbar:
        add_colorbar(ax, fig, norm,cmap,fmt_colorbar,title = title_cbar)
    
    return tmp

def add_colorbar(ax,fig,norm,cmap,fmt_colorbar = '%1.4f',title = None):
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.02, pad=0.04, aspect=50, format=fmt_colorbar)
    if title is not None : cbar.set_label(title)
    cbar.minorticks_off()


def get_figsize(document_width_pt = document_width_pt, wf=1, hf=0.5):
    """Parameters:
      - wf [float]:  width fraction in columnwidth units
      - hf [float]:  height fraction in columnwidth units.
                     Set by default to golden ratio.
      - columnwidth [float]: width of the column in latex. Var_sandwichet this from LaTeX 
                             using \showthe\columnwidth
    Returns:  [fig_width,fig_height]: that should be given to matplotlib
    """
    fig_width_pt = document_width_pt*wf
    inches_per_pt = 1.0/72.27               # Convert pt to inch
    fig_width = fig_width_pt*inches_per_pt  # width in inches
    fig_height = fig_width*hf      # height in inches
    return [fig_width, fig_height]



# Plots

def load_pi_r(sim_pi_r):
    pi_r = emp_pi_r
    df = pd.read_csv(input_folder / "X_dr.csv").query('X_dr > 0 & downstream_region')
    df['pi_r'] = df.X_dr/df.X_dr.sum()
    df["sim_pi_r"] = pi_r
    df.pi_r.fillna(0,inplace = True)
    df.sim_pi_r.fillna(0,inplace = True)
    return df

def plot_downstream(df,col_name,ax,fig):

    tmp = france.merge(df,on = "ze2010",how = "left")
    tmp[col_name].fillna(0,inplace = True)
    
    if col_name != "productivity":
        vmin = min(df.query('pi_r >0')["pi_r"].to_list())#+df.query('sim_pi_r >0')["sim_pi_r"].to_list())
        vmax = max(df["pi_r"].to_list()+df["sim_pi_r"].to_list())
        norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    else: 
        vmin,vmax = min(df.query('productivity >0').productivity),max(df['productivity'])
        norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
        print(df.productivity.describe())

    tmp.query(col_name+' == 0').plot(color = "gray",ax=ax)
    tmp.query(col_name+' > 0').plot(column = col_name,ax=ax,norm = norm, cmap = "viridis",edgecolor ="black")

    ax_inset = inset_axes(ax, width="65%", height="65%", loc="upper right",bbox_to_anchor=(0.745, 0.775, 0.25, 0.25), bbox_transform=ax.transAxes, borderpad=1)  
    data_idf = tmp[tmp['ze2010'].isin(idf_ze)]
    data_idf.query(col_name+' == 0').plot(color = "gray",ax=ax_inset)
    data_idf.query(col_name+' > 0').plot(column = col_name,ax=ax_inset,norm = norm, cmap = "viridis",edgecolor ="black")

    ax_inset.set_xticks([])
    ax_inset.set_yticks([])
    ax.set_xlim(-5, 10)
    ax.set_ylim(42, 52)
    ax.set_xticks([])
    ax.set_yticks([])

    add_colorbar(ax,fig,norm,cmap = "viridis")
    return norm


def load_productivity(productivity):
    prod = productivity
    df = filter_N_upstream_df[["ze2010","pi_r"]].drop_duplicates()
    df.loc[~df.pi_r.isna(),"productivity"] = prod
    df.productivity.fillna(0,inplace = True)
    return df



def unpack_simulated_moments(sim_moments, empirical_moments):
    keys = [
        "agg_labor_share",
        "agg_industry_share",
        "emp_gamma_ls",
        "reg_coef",
        "emp_pi_r"
    ]

    sizes = [m.size for m in empirical_moments]
    splits = np.cumsum(sizes)[:-1]
    blocks = np.split(sim_moments, splits, axis=0)

    result = {}
    for k, m, b in zip(keys, empirical_moments, blocks):
        reshaped = b.reshape(m.shape + (b.shape[1],))

        # Reconstruct first element for vectors that sum to 1 

        result[k] = reshaped

    return result
    
def bubble_scatter(
    ax, x, y, xlabel, ylabel, title,
    size_scale=300,
    regression_line=False,
    weighted_regression=False,
    weights=None
):
    mask = x > 0
    x, y = x[mask], y[mask]

    sizes = size_scale * x / x.max()
    lims = [min(x.min(), y.min()) * 1.2, max(x.max(), y.max()) * 1.2]

    data = pd.DataFrame({"x": x, "y": y})

    ax.scatter(
        x, y,
        s=sizes,
        alpha=0.6,
        edgecolor="black",
        color=toulouse_color
    )

    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(linestyle="dashed", alpha=0.5)

    # -------------------
    # Regression section
    # -------------------
    if weighted_regression:
        # Default weights: proportional to x (can be changed)
        if weights is None:
            weights = x

        model = sm.WLS.from_formula(
            'y ~ 0 + x',
            data=data,
            weights=weights
        )
    else:
        model = sm.OLS.from_formula('y ~ 0 + x', data=data)

    results = model.fit()
    b = results.params.iloc[0]

    ax.text(
        0.98, 0.09,
        fr'Coefficient: ${np.round(b, 3)}$',
        ha='right', va='bottom',
        fontsize=10,
        transform=ax.transAxes
    )

    ax.text(
        0.98, 0.01,
        fr't-stat: ${np.round(results.tvalues.iloc[0], 1)}$',
        ha='right', va='bottom',
        fontsize=10,
        transform=ax.transAxes
    )

    # -------------------
    # Lines
    # -------------------
    if regression_line:
        X = np.linspace(0, lims[1], 100)
        Y = b * X
        ax.plot(X, Y, linestyle='--', color='green')
    else:
        ax.plot(lims, lims, color="black")

    sns.despine()



In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
from pathlib import Path


def load_industry_data(industry,gmm = False):
    """
    Load all necessary data for a given industry.
    
    Parameters:
    -----------
    industry : str
        Industry code (e.g., "auto", "aero")
    
    Returns:
    --------
    dict with all loaded data and derived variables
    """
    
    # =========================================================================
    # PATHS
    # =========================================================================
    gmm = "_gmm" if gmm else ""
    input_folder = Path(f"../baseline_{industry}")
    folder = Path(f"../reporting{gmm}_{industry}/")
    n_coef = np.load(os.path.join(folder,"n_reg_coef.npy"))
    gamma_threshold = np.load(os.path.join(folder,"gamma_threshold.npy"))
    
    # Downstream sector
    d = "C30C" if industry == "aero" else "C29A"
    
    # =========================================================================
    # COEFFICIENTS
    # =========================================================================
    coefs = pd.read_csv(input_folder / "stats.csv")
    agg_labor_share = coefs.loc[1, "value"]
    epsilon = coefs.loc[0, "value"]
    
    # =========================================================================
    # ARRAYS
    # =========================================================================
    
    agg_industry_share = np.load(input_folder / "input_share.npy")
    emp_gamma_ls = np.load(input_folder / "emp_gamma_ls.npy")
    emp_gamma_ls[emp_gamma_ls/np.sum(emp_gamma_ls, axis = 1,keepdims=True)<= gamma_threshold] = 0 
    emp_pi_r = pd.read_csv(input_folder / "X_dr.csv").query('X_dr > 0 & downstream_region').X_dr.values
    emp_pi_r = (emp_pi_r/emp_pi_r.sum())
    reg_coef = np.load(input_folder / f"reg_coef_4.npy") if n_coef == 4 else np.array(coefs.loc[coefs.stats == "reg_coef",'value'].values[0])
    N_downstream_per_region_local = np.load(os.path.join(input_folder,"N_downstream_per_region.npy"))
    regional_wage = np.load(input_folder / "regional_wages.npy")
    X_dr = pd.read_csv(os.path.join(input_folder / "X_dr.csv"))
    X_dr['ze2010'] = X_dr['ze2010'].astype(str).str.zfill(4)

    
    # =========================================================================
    # REFERENCE EMPIRICAL MOMENTS
    # =========================================================================
    reference_empirical_moments = [
        np.array([agg_labor_share]),
        agg_industry_share,
        emp_pi_r,
        reg_coef,
        emp_gamma_ls
    ]
    empirical_moments = np.concatenate([np.ndarray.flatten(d) for d in reference_empirical_moments]).reshape(-1,1)

    # =========================================================================
    # ILE-DE-FRANCE ZONES
    # =========================================================================
    idf_ze = [
        '1101', '1111', '1102', '1104', '1118', '1115', '1116', '1105',
        '1117', '1110', '1119', '1112', '1103', '1109', '1106', '1114',
        '1113', '1108', '1107'
    ]
    
    # =========================================================================
    # FILTER DATA
    # =========================================================================
    filter_N_upstream_df = pd.read_csv(input_folder / "filter_N_upstream.csv")
    filter_N_upstream_df['ze2010'] = filter_N_upstream_df['ze2010'].astype(str).str.zfill(4)
    
    # =========================================================================
    # DISTANCES AND GEOGRAPHY
    # =========================================================================
    distances = np.load(input_folder / "full_distances.npy")   
    
    france = gpd.read_file(input_folder / "france.gpkg", encoding='utf-8').sort_values(by='ze2010')

    
    # Build distance reference dataframe
    ref = pd.DataFrame(
        distances[:297, :297], 
        index=france["ze2010"].values, 
        columns=france["ze2010"].values
    )
    ref.reset_index(inplace=True)
    ref.rename(columns={'index': 'ze2010_i'}, inplace=True)
    ref = ref.melt(id_vars='ze2010_i', var_name='ze2010_j', value_name='M_ij')
    
    # =========================================================================
    # SIMULATED MOMENTS
    # =========================================================================
    best_simulated_moments = np.load(folder / "best_simulated_moments.npy")
    best_params = np.load(folder / "best_parameters_list.npy")
    try:
        panel_df = pd.read_parquet(folder/ "simulated_panel_unified.parquet")
        regional_panel_df = pd.read_parquet(folder/ "regional_sales_unified.parquet")
        suppliers = pd.read_parquet(folder / "suppliers.parquet")
        
    except : 
        panel_df = None
        regional_panel_df = None
        suppliers = None
    # Unpack moments
    best_simulated_moments_dict = unpack_simulated_moments(
        best_simulated_moments, 
        reference_empirical_moments
    )
    empirical_moments_dict = dict(zip([
        "agg_labor_share",
        "agg_industry_share",
        "emp_pi_r",
        "reg_coef",
        "emp_gamma_ls"
    ], reference_empirical_moments))
    
    # =========================================================================
    # RETURN ALL DATA
    # =========================================================================
    return {
        # Paths
        'input_folder': input_folder,
        'folder': folder,
        
        # Industry info
        'industry': industry,
        'industry': industry,
        'd': d,
        
        # Coefficients
        'coefs': coefs,
        'agg_labor_share': agg_labor_share,
        'epsilon': epsilon,
        'n_coef':n_coef,
        
        # Arrays
        'agg_industry_share': agg_industry_share,
        'emp_gamma_ls': emp_gamma_ls,
        'emp_pi_r': emp_pi_r,
        'reg_coef': reg_coef,
        "regional_wage":regional_wage,
        "X_dr":X_dr,
        
        # Moments
        'reference_empirical_moments': reference_empirical_moments,
        'empirical_moments': empirical_moments,
        'best_simulated_moments': best_simulated_moments,
        'best_params': best_params,
        'best_simulated_moments_dict': best_simulated_moments_dict,
        'empirical_moments_dict': empirical_moments_dict,

        # Regression 
        'panel_df':panel_df,
        "regional_panel_df":regional_panel_df,
        "suppliers":suppliers,
        
        # Geography
        'idf_ze': idf_ze,
        'filter_N_upstream_df': filter_N_upstream_df,
        "N_downstream":N_downstream_per_region_local,
        'distances': distances,
        'france': france,
        'ref': ref,
    }


# =============================================================================
# EXAMPLE USAGE
# =============================================================================
industry = "auto"
# Load data
data = load_industry_data(industry,False)

# Unpack variables (optional - can also access via data['variable_name'])
input_folder = data['input_folder']
folder = data['folder']
d = data['d']
agg_labor_share = data['agg_labor_share']
epsilon = data['epsilon']
agg_industry_share = data['agg_industry_share']
emp_gamma_ls = data['emp_gamma_ls']
emp_pi_r = data['emp_pi_r']
reg_coef = data['reg_coef']
reference_empirical_moments = data['reference_empirical_moments']
idf_ze = data['idf_ze']
filter_N_upstream_df = data['filter_N_upstream_df']
distances = data['distances']
france = data['france']
ref = data['ref']
n_coef = data['n_coef']
empirical_moments = data['empirical_moments']
best_simulated_moments = data['best_simulated_moments']
best_params = data['best_params']
best_simulated_moments_dict = data['best_simulated_moments_dict']
empirical_moments_dict = data['empirical_moments_dict']
panel_df = data['panel_df']
regional_panel_df = data['regional_panel_df']
suppliers = data['suppliers']
N_downstream = data['N_downstream']
X_dr = data['X_dr']

# Or use a helper to unpack all at once
def unpack_data(data_dict):
    """Unpack data dictionary into global namespace."""
    return data_dict.values()

# Or more elegantly with locals().update() in a notebook:
# locals().update(data)

In [ ]:

def unpack_simulated_moments(sim_moments, empirical_moments):
    keys = [
        "agg_labor_share",
        "agg_industry_share",
        "emp_pi_r",
        "reg_coef",
        "emp_gamma_ls"
    ]

    sizes = [m.size for m in empirical_moments]
    splits = np.cumsum(sizes)[:-1]
    blocks = np.split(sim_moments, splits, axis=0)

    result = {}
    for k, m, b in zip(keys, empirical_moments, blocks):
        reshaped = b.reshape(m.shape + (b.shape[1],))

        # Reconstruct first element for vectors that sum to 1 

        result[k] = reshaped

    return result

# Reporting

## Merging reporting tables

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Assuming unpack_simulated_moments is defined elsewhere in your codebase
# from your_module import unpack_simulated_moments


def generate_combined_table(industries_config, output_file="moments_comparison_combined.tex"):
    """
    Generate a combined LaTeX table with subcolumns for multiple industries.
    
    Parameters:
    -----------
    industries_config : list of dict
        Each dict contains: 'industry', 'industry', 'display_name'
        Example: [{'industry': 'auto', 'industry': 'auto', 'display_name': 'Car'},
                  {'industry': 'aero', 'industry': 'aero', 'display_name': 'Aerospace'}]
    output_file : str
        Path to the output LaTeX file.
    n_reg_coef : int
        Number of regression coefficients to display (4 or 5). 
        If 4, the ]20,50] bin is excluded. Default is 5.
    """
    K = -1  # Use last iteration
    
    # Load data for all industries
    all_data = {}
    for config in industries_config:
        industry = config['industry']
        industry = config['industry']
        all_data[industry] = load_industry_data(industry)

    n_reg_coef = len(all_data['aero']['reg_coef'])
    # Load industry names
    name_A129 = pd.read_csv("../external/A129_name_fr_eng.csv")
    
    # Number of industries
    n_industries = len(industries_config)
    
    # Build column spec for siunitx
    col_spec = "l " + " ".join(["S[table-format=1.4] S[table-format=1.4]" for _ in range(n_industries)])
    
    # Build header with multicolumn for each industry
    header_row1 = " & " + " & ".join([f"\\multicolumn{{2}}{{c}}{{{config['display_name']}}}" for config in industries_config]) + " \\\\"
    
    # Cmidrule for each industry pair
    cmidrules = ""
    for i, _ in enumerate(industries_config):
        start_col = 2 + i * 2
        end_col = start_col + 1
        cmidrules += f"\\cmidrule(lr){{{start_col}-{end_col}}} "
    
    header_row2 = " & " + " & ".join(["{Emp.} & {Sim.}" for _ in industries_config]) + " \\\\"
    
    # === Panel A: Aggregate Labor Share ===
    panel_A_rows = []
    row_data = ["Aggregate Labor Share"]
    for config in industries_config:
        industry = config['industry']
        data = all_data[industry]
        emp_val = np.round(data['agg_labor_share'], 4)
        sim_val = np.round(data['best_simulated_moments_dict']['agg_labor_share'][:, K][0], 4)
        row_data.extend([emp_val, sim_val])
    panel_A_rows.append(row_data)
    
    # === Panel B: Aggregate Industry Shares ===
    # Get union of all A129 codes across industries
    all_A129_codes = set()
    for config in industries_config:
        industry = config['industry']
        data = all_data[industry]
        codes = list(data['filter_N_upstream_df']['A129'].drop_duplicates().sort_values())
        all_A129_codes.update(codes)
    all_A129_codes = sorted(all_A129_codes)
    
    # Create industry share dataframes for each industry
    industry_shares = {}
    for config in industries_config:
        industry = config['industry']
        data = all_data[industry]
        A129_codes = list(data['filter_N_upstream_df']['A129'].drop_duplicates().sort_values())
        sim_shares = list(data['best_simulated_moments_dict']['agg_industry_share'][:, K])
        industry_shares[industry] = pd.DataFrame({
            "A129": A129_codes,
            "Empirical": list(np.round(data['agg_industry_share'], 4)),
            "Simulated": np.round(sim_shares, 4)
        })
    
    # Build Panel B rows
    panel_B_rows = []
    for code in all_A129_codes:
        sector_name = name_A129[name_A129['A129'] == code]['name'].values
        sector_name = sector_name[0] if len(sector_name) > 0 else "Non-upstream sector"
        
        row_data = [sector_name]
        for config in industries_config:
            industry = config['industry']
            df = industry_shares[industry]
            match = df[df['A129'] == code]
            if len(match) > 0:
                emp_val = match.iloc[0]['Empirical']
                sim_val = match.iloc[0]['Simulated']
            else:
                emp_val, sim_val = "---", "---"
            row_data.extend([emp_val, sim_val])
        panel_B_rows.append(row_data)
    
    # === Panel C: Regression Coefficients ===
    # Define coefficient names based on n_reg_coef parameter
    all_coef_names_full = [r'$]20,50]$', r'$]50,100]$', r'$]100,150]$', r'$]150,200]$', r'$>200$']
    
    if n_reg_coef == 4:
        # Exclude the first bin (]20,50])
        all_coef_names = all_coef_names_full[1:]
    else:
        # Use all 5 bins
        all_coef_names = all_coef_names_full
    
    panel_C_rows = []
    for i, coef_name in enumerate(all_coef_names):
        row_data = [coef_name]
        for config in industries_config:
            industry = config['industry']
            data = all_data[industry]
            
            # Determine which coefficient index to use
            coef_idx = i
            
            if coef_idx < len(data['reg_coef']):
                emp_val = np.round(data['reg_coef'][coef_idx], 3)
                sim_val = np.round(data['best_simulated_moments_dict']['reg_coef'][:, K][coef_idx], 3)
            else:
                emp_val, sim_val = "---", "---"
            row_data.extend([emp_val, sim_val])
        panel_C_rows.append(row_data)
    
    # === Generate LaTeX Table ===
    def format_value(val, decimals=4):
        """Format a value for LaTeX, handling '---' for missing data."""
        if val == "---" or (isinstance(val, float) and np.isnan(val)):
            return "{---}"
        return f"{val:.{decimals}f}"
    
    latex_table = r"""\begin{table}[H]
\centering
\caption{Empirical and Simulated Moments: Comparison Across Industries}
\label{tab:moments_comparison_combined}
\renewcommand{\arraystretch}{1.2}
\small
\begin{tabular}{""" + col_spec + r"""}
\toprule
""" + header_row1 + "\n" + cmidrules + "\n" + header_row2 + "\n"
    
    # Panel A
    latex_table += "\\midrule\n"
    latex_table += f"\\multicolumn{{{1 + 2*n_industries}}}{{l}}{{\\textbf{{Panel A: Aggregate Labor Share}}}} \\\\\n"
    latex_table += "\\midrule\n"
    for row in panel_A_rows:
        row_str = row[0] + " & " + " & ".join([format_value(v, 4) for v in row[1:]]) + " \\\\\n"
        latex_table += row_str
    
    # Panel B
    latex_table += "\\midrule\n"
    latex_table += f"\\multicolumn{{{1 + 2*n_industries}}}{{l}}{{\\textbf{{Panel B: Aggregate Industry Shares}}}} \\\\\n"
    latex_table += "\\midrule\n"
    for row in panel_B_rows:
        row_str = row[0] + " & " + " & ".join([format_value(v, 4) for v in row[1:]]) + " \\\\\n"
        latex_table += row_str
    
    # Panel C
    latex_table += "\\midrule\n"
    latex_table += f"\\multicolumn{{{1 + 2*n_industries}}}{{l}}{{\\textbf{{Panel C: Regression Coefficients}}}} \\\\\n"
    latex_table += "\\midrule\n"
    for row in panel_C_rows:
        row_str = row[0] + " & " + " & ".join([format_value(v, 3) for v in row[1:]]) + " \\\\\n"
        latex_table += row_str
    
    latex_table += "\\bottomrule\n"
    latex_table += r"""\end{tabular}

\vspace{0.3cm}
\caption*{\footnotesize \emph{Notes}: This table compares empirical moments with moments simulated from the model calibrated on each industry. ``Emp.'' = Empirical, ``Sim.'' = Simulated. Panel~A shows the aggregate labor share, Panel~B shows sectoral industry shares, and Panel~C shows regression coefficients estimated empirically using Equation~\ref{eq:reduced_form_spatialIO} on finer distance bins. ``---'' indicates the moment is not applicable for that industry. Shares are rounded to 4 decimals, coefficients to 3 decimals.}
\end{table}
"""
    
    # Save to file
    with open(output_file, "w") as f:
        f.write(latex_table)
    
    print(f"Combined table saved to: {output_file}")
    return latex_table


# === Main execution ===
if __name__ == "__main__":
    # Define industries to compare
    industries_config = [
        {'industry': 'auto', 'industry': 'auto', 'display_name': 'Motor Vehicles'},
        {'industry': 'aero', 'industry': 'aero', 'display_name': 'Aerospace'}
    ]
    
    # Generate combined table
    output_folder = "../reporting_combined/"
    os.makedirs(output_folder, exist_ok=True)
    
    latex_output = generate_combined_table(
        industries_config,
        output_file=os.path.join(output_folder, "moments_comparison_combined.tex")
    )
    
    print(latex_output)

## Table, maps and scatter plots

In [ ]:

for industry in ['auto','aero']:
    data = load_industry_data(industry)
    globals().update(data)  # In a script

    emp_g = empirical_moments_dict['emp_gamma_ls'].ravel()
    sim_g = best_simulated_moments_dict['emp_gamma_ls'][:, :, K].ravel()
    sim_g = sim_g[emp_g!=0]
    emp_g = emp_g[emp_g!=0]
            
    fig, axs = plt.subplots(1, 1, figsize=get_figsize())

    bubble_scatter(
        axs,
        emp_g,
        sim_g,
        r"Empirical",
        r"Simulated",
        None,regression_line = False
    )

    fig.savefig(os.path.join("../reporting_combined", f'emp_sim_gamma_{industry}.pdf'), format='pdf', bbox_inches='tight')

    fig, axs = plt.subplots(1, 1, figsize=get_figsize())


    emp_pi_r = empirical_moments_dict['emp_pi_r'].ravel()
    sim_pi_r = best_simulated_moments_dict['emp_pi_r'][:, K].ravel()

    bubble_scatter(
        axs,
        emp_pi_r,
        sim_pi_r,
        r"Empirical",
        r"Simulated",
        None,
        size_scale=400,regression_line = False
    )
    fig.savefig(os.path.join("../reporting_combined", f'emp_sim_pi_{industry}.pdf'), format='pdf', bbox_inches='tight')


## Jacobian and variance covariance

### Jacobian decomposition

In [ ]:

bool_gmm = False
industry = "auto"
data = load_industry_data(industry,bool_gmm)
globals().update(data)  
suffix = "2x2_ana_fd_check" #"all_step3" #"2x2_ana"#2x2_ana_fd_check
gmm = "_gmm" if bool_gmm else ""
# La Jacobienne est définie sur tous les paramètres sauf ceux de référence.
# De même pour les moments.  
J = np.load(f"../reporting{gmm}_{industry}/step3/jacobian_{suffix}.npy")
J_sd = np.load(f"../reporting{gmm}_{industry}/step3/jacobian_{suffix}_sd.npy")
J_e = np.load(f"../reporting{gmm}_{industry}/step3/jacobian_{suffix}_elasticity.npy")
J_e_sd = np.load(f"../reporting{gmm}_{industry}/step3/jacobian_{suffix}_elasticity_sd.npy")


n_labor    = 1 
n_industry = empirical_moments_dict['agg_industry_share'].shape[0] - 1
n_gamma    = np.count_nonzero(np.ndarray.flatten(empirical_moments_dict['emp_gamma_ls'])) - (n_industry + 1) 
n_reg      = n_coef 
n_pi_r     = empirical_moments_dict['emp_pi_r'].shape[0] - 1

sector_names = filter_N_upstream_df[['A129']].drop_duplicates().sort_values(by = "A129").reset_index()
sector_names.index +=1 
sector_names = sector_names['A129'].to_dict()

ze_names = filter_N_upstream_df[['ze2010']].drop_duplicates().sort_values(by = "ze2010").reset_index()
ze_names.index +=1 
ze_names = ze_names['ze2010'].to_dict()

gamma_matrix = empirical_moments_dict['emp_gamma_ls']
index_labels = [sector_names[i] for i in range(1, len(sector_names) + 1)]
column_labels = [ze_names[i] for i in range(1, len(ze_names) + 1)]

# 2. Create the DataFrame
df_gamma = pd.DataFrame(
    gamma_matrix, 
    index=index_labels, 
    columns=column_labels
)

# Display result
df_gamma = df_gamma.stack().reset_index()
df_gamma = df_gamma.reset_index()
df_gamma.columns = ['coord','A129',"ze2010","gamma_ls"]
df_gamma.sort_values(by = ['A129','ze2010'],inplace = True)
df_gamma = df_gamma.query('gamma_ls != 0')
df_gamma['new_coord'] = df_gamma.coord.rank().astype(int)+ n_reg -1
df_gamma['params'] = df_gamma['A129'] + "-" + df_gamma['ze2010']
df_gamma['reference'] = df_gamma.gamma_ls == df_gamma.groupby('A129').gamma_ls.transform('max')


y_ticks_map = {
    "labor":None,
    "industry":[x for x in sector_names.values()],
    "gamma_ls":df_gamma.params.to_list(),
    "reg_coef":["beta_1","beta_2","beta_3","beta_4"],
    "pi_r":X_dr.query('downstream_region').ze2010.to_list(),
}
beta_coef =  ["beta_1","beta_2","beta_3","beta_4"] if n_coef == 4 else ['beta']
df = pd.DataFrame(["labor"] + y_ticks_map["industry"][1:] + y_ticks_map["pi_r"][1:] + beta_coef, columns = ['params'])
df = pd.concat([df,df_gamma.query('~reference')[['params']]])
df = df.reset_index(drop = True).reset_index()
df['coord'] = df['index'].rank().astype(int)-1
df['label'] = df.params
df = df[['label','coord']]
df.head(50)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap


y_ticks_map = {

    "Extensive margin":["alpha_1","alpha_2","alpha_3","alpha_4"] if n_coef == 4 else ['alpha'],
    "Labor share":['Labor'],
    "Industry shares":[x for x in sector_names.values()][1:],
    "Downstream sales":X_dr.query('downstream_region').ze2010.to_list()[1:],
    "Regional sourcing shares":df_gamma.query('~reference').params.to_list()
}
# --- 1. Prepare Data ---
block_sizes_moment = [n_labor, n_industry, n_pi_r, n_coef, n_gamma]
block_sizes_parameters = [n_labor, n_industry, n_pi_r, n_tau, n_gamma]
block_names = ["Labor share", "Industry shares", "Downstream sales", "Extensive margin", "Regional sourcing shares"]
block_edges_moments = np.cumsum([0] + block_sizes_moment)
block_edges_parameters = np.cumsum([0] + block_sizes_parameters)
split_indices_moments = np.cumsum(block_sizes_moment)[:-1]
split_indices_parameters = np.cumsum(block_sizes_parameters)[:-1]
J_blocks_list = np.split(J_e, split_indices_moments,axis = 0)

# --- 2. SETUP BINNED COLORMAP ---
# Your specific thresholds
# We add -inf and inf to ensure values outside the range are still colored
boundaries = [-1, -0.1, -0.05, -0.025, -0.01, -0.001, 0.001, 0.01, 0.025, 0.05, 0.1, 1]

# Create a discrete colormap. 
# We take the 'RdBu_r' map and sample colors at the midpoints of your bins
# to ensure we get a good transition from Red -> White -> Blue.
n_bins = len(boundaries) - 1
cmap_base = plt.get_cmap('RdBu_r')
# We sample colors from the continuous map to create a discrete one
colors = cmap_base(np.linspace(0, 1, n_bins))
binned_cmap = ListedColormap(colors)

# BoundaryNorm maps the data into the indices of the colormap
norm = BoundaryNorm(boundaries, n_bins)

# --- 3. LOOP AND PLOT ---
for i in range(len(block_names)):
    name = block_names[i]
    data = J_blocks_list[i]
    
    fig, ax = plt.subplots(figsize=(8, 6))
    # Alternate subplot background shading
    
    # --- PLOTTING ---
    # Use the new binned_cmap and BoundaryNorm
    im = ax.matshow(data, cmap=binned_cmap, norm=norm, aspect='auto')
    
    # Add the horizontal colorbar
    # We use the boundaries as ticks so the user sees where bins change
    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.05, pad=0.15, ticks=boundaries)
    
    # Format labels to be clean
    cbar.ax.set_xticklabels([f'{t:g}' for t in boundaries], fontsize=8, rotation=45)
    cbar.set_label('Elasticity Magnitude (Binned)', fontsize=10)

    # --- FORMATTING ---
    ax.set_title(f"Jacobian Block: {name.upper()}\nShape: {data.shape[0]} $\\times$ {data.shape[1]}", 
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel("Parameter Index", fontsize=12)
    ax.set_ylabel("Moment Index", fontsize=12)
    
    # X-axis tick adjustment

    # --- X-AXIS TICK ADJUSTMENT ---
    # 1. Define which tick positions we want (e.g., every 10th index)
    tick_positions = np.arange(data.shape[1])[::10]
    
    # 2. Define the labels for those specific positions
    # Note: ensure 'df.label' is a list/array that matches the full length of parameters
    tick_labels = df.label.values[tick_positions]  

    # 3. Apply ticks and labels
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, rotation=90, fontsize=8) 
    
    # --- Y-AXIS TICK LOGIC ---
    labels = y_ticks_map.get(name)

    if labels is not None:
        # Determine positions and labels based on the block type
        labels = np.array(labels)
        if name == "Regional sourcing shares":
            # Sampling: 1 out of every 10
            tick_positions = np.arange(len(labels))[::10]
            tick_labels = labels[tick_positions]
        else:
            # For other blocks, we show all labels 
            # (Or you can add [::5] if they are still too crowded)
            tick_positions = np.arange(len(labels))
            tick_labels = labels

        # Apply the ticks
        ax.set_yticks(tick_positions)
        
        # Apply labels with alignment
        # We use rotation=0 for Y-axis labels for better readability
        ax.set_yticklabels(tick_labels, fontsize=8)
        
        # Ensure labels are aligned to the right of the axis
        ax.yaxis.set_tick_params(pad=5) 

    else:
        # If labels is None (like for 'labor'), we can just hide the labels 
        # or show numbers. Usually, hiding is cleaner for single-row blocks.
        ax.set_yticklabels([])

    
    # --- BLOCK SEPARATORS ---
    for edge in block_edges_parameters[1:-1]:

        ax.axvline(
            edge - 0.5,
            color='black',
            linewidth=1.2,
            alpha=0.5
        )
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap


# --- 2. SETUP BINNED COLORMAP ---
boundaries = [-1, -0.1, -0.05, -0.025, -0.01, -0.001, 0.001, 0.01, 0.025, 0.05, 0.1, 1]
n_bins = len(boundaries) - 1
cmap_base = plt.get_cmap('RdBu_r')
colors = cmap_base(np.linspace(0, 1, n_bins))
binned_cmap = ListedColormap(colors)
norm = BoundaryNorm(boundaries, n_bins)

# --- 2. SETUP BINNED COLORMAP (HIVar_sandwichHLIVar_sandwichHTING EXTREMES) ---

# We define boundaries such that the middle range [-0.1, 0.1] is one single bin
# The bins will be: [-1, -0.1] (Red/Blue), [-0.1, 0.1] (Gray), [0.1, 1] (Red/Blue)
boundaries = [-1, -0.1, 0.1, 1]
n_bins = len(boundaries) - 1

# Create a custom color list:
# Bin 0: [-1 to -0.1] -> Dark Red
# Bin 1: [-0.1 to 0.1] -> Light Gray (The "Neutral" zone)
# Bin 2: [0.1 to 1] -> Dark Blue
colors = ['#2166ac', '#f0f0f0','#b2182b'] 

binned_cmap = ListedColormap(colors)
norm = BoundaryNorm(boundaries, n_bins)

# --- 3. SETUP SUBPLOT GRID ---
# Using 2 rows and 3 columns to accommodate 5 blocks
fig, axes = plt.subplots(2, 3, figsize=(18, 12), constrained_layout=True)
axes = axes.flatten() # Convert 2D grid to 1D list for easy looping

# --- 4. LOOP AND PLOT ---
for i in range(len(block_names)):
    name = block_names[i]
    data = J_blocks_list[i]
    ax = axes[i]
    
    # --- BLOCK SEPARATORS ---
    for edge in block_edges_parameters[1:-1]:

        ax.axvline(
            edge - 0.5,
            color='black',
            linewidth=1.2,
            alpha=0.5
        )
    # --- PLOTTING ---
    im = ax.matshow(data, cmap=binned_cmap, norm=norm, aspect='auto')
    
    # --- INDIVIDUAL COLORBAR PER SUBPLOT ---
    # We attach it to the current 'ax'
    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', fraction=0.05, pad=0.15, ticks=boundaries)
    cbar.ax.set_xticklabels([f'{t:g}' for t in boundaries], fontsize=7, rotation=45)
    cbar.set_label('Elasticity', fontsize=9)

    # --- FORMATTING ---
    ax.set_title(f"Block: {name.upper()}\n({data.shape[0]}x{data.shape[1]})", 
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel("Parameter Index", fontsize=10)
    ax.set_ylabel("Moment Index", fontsize=10)
    
    # --- X-AXIS TICK ADJUSTMENT ---
    # Get positions and labels from df.label
    x_tick_pos = np.arange(data.shape[1])[::10]
    x_tick_labels = df.label.values[x_tick_pos] 

    ax.set_xticks(x_tick_pos)
    ax.set_xticklabels(x_tick_labels, rotation=90, fontsize=8) 
    
    # --- Y-AXIS TICK LOGIC ---
    labels_raw = y_ticks_map.get(name)

    if labels_raw is not None:
        labels = np.array(labels_raw)
        if name == "Regional sourcing shares":
            # Sampling: 1 out of every 10
            y_tick_pos = np.arange(len(labels))[::10]
            y_tick_labels = labels[y_tick_pos]
        else:
            y_tick_pos = np.arange(len(labels))
            y_tick_labels = labels

        ax.set_yticks(y_tick_pos)
        ax.set_yticklabels(y_tick_labels, fontsize=8)
        ax.yaxis.set_tick_params(pad=5) 
    else:
        ax.set_yticks([]) # Hide ticks for blocks like 'labor'

# --- 5. CLEANUP UNUSED SUBPLOTS ---
# Since we have 6 subplots (2x3) but only 5 blocks, hide the 6th one
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle("Jacobian Elasticity Decomposition by Moment Blocks", fontsize=20, fontweight='bold')
plt.show()